# Stratigraphic State Feature Screen

This notebook tests whether hidden-test-safe stratigraphic-state features can recover part of the oracle formation-distance signal seen in prior XGBoost diagnostics. It uses known-zone TVT/GR/trajectory summaries, typewell log-alignment uncertainty, and fold-safe nearest-analog descriptors.

## 1. Purpose And Promotion Gates

Promotion requires improvement over the current inference-safe XGBoost local reference on `strict_hybrid_fold`, no regression on `analog_distance_fold`, and visible gains in stress slices such as poor alignment, high extrapolation, or high analog novelty. Direct train-only formation columns remain oracle diagnostics only.

In [1]:
"Configure imports, paths, and run constants."

from pathlib import Path
import json
import math
import shutil
import time

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
except Exception as exc:
    XGB_AVAILABLE = False
    XGB_IMPORT_ERROR = repr(exc)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "context").exists():
    PROJECT_ROOT = next(parent for parent in PROJECT_ROOT.parents if (parent / "context").exists())

import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.io.loaders import discover_wells, load_horizontal_well, load_typewell
from src.preprocessing.stratigraphic import (
    alignment_uncertainty,
    best_lag_correlation,
    dtw_distance,
    resample_sequence,
    robust_standardize,
    summarize_known_zone_state,
)

DATE_STAMP = "2026-06-30"
RUN_ID = f"{DATE_STAMP}__stratigraphic_state_feature_screen"
DATA_ROOT = PROJECT_ROOT / "data" / "raw"
TRAIN_DIR = DATA_ROOT / "train"
FOLD_PATH = PROJECT_ROOT / "artifacts" / "tables" / "validation__strict_cv_fold_assignments__2026-06-26.csv"
ARTIFACT_TABLES = PROJECT_ROOT / "artifacts" / "tables"
ARTIFACT_REPORTS = PROJECT_ROOT / "artifacts" / "reports"
ARTIFACT_LOGS = PROJECT_ROOT / "artifacts" / "logs" / "runs" / RUN_ID
for out_dir in [ARTIFACT_TABLES, ARTIFACT_REPORTS, ARTIFACT_LOGS]:
    out_dir.mkdir(parents=True, exist_ok=True)

MAX_TARGET_ROWS_PER_WELL = 90
SEQUENCE_LENGTH = 32
TYPEWELL_WINDOWS = 18
RANDOM_STATE = 42
CURRENT_PUBLIC_BEST = 14.472
LOCAL_SAFE_XGB_REFERENCE = 13.4178

print(f"project_root={PROJECT_ROOT}")
print(f"run_id={RUN_ID}")
print(f"xgboost_available={XGB_AVAILABLE}")

project_root=C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling
run_id=2026-06-30__stratigraphic_state_feature_screen
xgboost_available=True


## 2. Data Loading

In [2]:
"Load train wells and strict fold assignments."

folds = pd.read_csv(FOLD_PATH)
fold_cols = ["strict_hybrid_fold", "analog_distance_fold"]
wellnames = discover_wells(TRAIN_DIR)
fold_wells = set(folds["wellname"].astype(str))
wellnames = [well for well in wellnames if well in fold_wells]

well_data = {}
for well in wellnames:
    horizontal = load_horizontal_well(TRAIN_DIR / f"{well}__horizontal_well.csv")
    typewell_path = TRAIN_DIR / f"{well}__typewell.csv"
    typewell = load_typewell(typewell_path) if typewell_path.exists() else pd.DataFrame()
    if "TVT" not in horizontal.columns:
        continue
    well_data[well] = {"horizontal": horizontal, "typewell": typewell}

print(f"loaded_wells={len(well_data)}")
print(f"fold_rows={len(folds)}")

11:16:29 | INFO     | src.io.loaders | Discovered 773 wells in C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\data\raw\train


loaded_wells=773
fold_rows=773


## 3. Fold-Safe Stratigraphic Feature Builders

In [3]:
"Define row, typewell-alignment, and analog feature builders."

FORMATION_COLS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]


def _safe_gr(series):
    gr = series.astype(float).clip(lower=0, upper=300)
    return gr.fillna(gr.rolling(21, center=True, min_periods=1).median()).ffill().bfill().fillna(0.0)


def _target_rows(df):
    mask = df["TVT"].notna() & df["TVT_input"].isna()
    return df.loc[mask].copy()


def _sample_target_rows(df, max_rows=MAX_TARGET_ROWS_PER_WELL):
    if len(df) <= max_rows:
        return df
    idx = np.linspace(0, len(df) - 1, max_rows).astype(int)
    return df.iloc[idx].copy()


def _known_frame(df):
    return df[df["TVT_input"].notna()].copy()


def build_typewell_alignment_features(horizontal, typewell):
    known = _known_frame(horizontal)
    known_gr = _safe_gr(known["GR"]) if not known.empty and "GR" in known else pd.Series(dtype=float)
    known_seq = resample_sequence(known_gr.to_numpy(), SEQUENCE_LENGTH)
    if typewell.empty or "GR" not in typewell:
        base = alignment_uncertainty([])
        base.update({
            "typewell_best_lag": 0.0,
            "typewell_best_corr": 0.0,
            "typewell_second_corr": 0.0,
            "typewell_best_dtw": 0.0,
            "typewell_window_position": 0.0,
        })
        return base

    tw_gr = _safe_gr(typewell["GR"]).to_numpy()
    if len(tw_gr) < 4:
        scores = [0.0]
        dtws = [0.0]
        positions = [0.0]
    else:
        window_count = min(TYPEWELL_WINDOWS, max(1, len(tw_gr) // 4))
        starts = np.linspace(0, max(0, len(tw_gr) - SEQUENCE_LENGTH), window_count).astype(int)
        scores, dtws, positions = [], [], []
        for start in starts:
            raw_window = tw_gr[start : start + SEQUENCE_LENGTH]
            window = resample_sequence(raw_window, SEQUENCE_LENGTH)
            lag = best_lag_correlation(known_seq, window, max_lag=6)
            dist = dtw_distance(known_seq, window)
            score = lag["best_corr"] - 0.05 * dist
            scores.append(score)
            dtws.append(dist)
            positions.append(start / max(len(tw_gr) - 1, 1))
    best_idx = int(np.nanargmax(scores)) if len(scores) else 0
    best_window = resample_sequence(tw_gr[int(positions[best_idx] * max(len(tw_gr) - 1, 1)) : int(positions[best_idx] * max(len(tw_gr) - 1, 1)) + SEQUENCE_LENGTH], SEQUENCE_LENGTH) if len(tw_gr) else np.zeros(SEQUENCE_LENGTH)
    lag = best_lag_correlation(known_seq, best_window, max_lag=6)
    unc = alignment_uncertainty(scores)
    unc.update({
        "typewell_best_lag": lag["best_lag"],
        "typewell_best_corr": lag["best_corr"],
        "typewell_second_corr": lag["second_corr"],
        "typewell_best_dtw": float(dtws[best_idx]),
        "typewell_window_position": float(positions[best_idx]),
    })
    return unc


def build_well_descriptor(well):
    horizontal = well_data[well]["horizontal"]
    typewell = well_data[well]["typewell"]
    known_state = summarize_known_zone_state(horizontal)
    align_state = build_typewell_alignment_features(horizontal, typewell)
    target = _target_rows(horizontal)
    known = _known_frame(horizontal)
    if target.empty or known.empty:
        extra = {"target_row_count": float(len(target)), "target_md_span": 0.0, "endpoint_gap_to_target": 0.0}
    else:
        extra = {
            "target_row_count": float(len(target)),
            "target_md_span": float(target["MD"].iloc[-1] - target["MD"].iloc[0]),
            "endpoint_gap_to_target": float(target["MD"].iloc[0] - known["MD"].iloc[-1]),
        }
    return {"wellname": well, **known_state, **align_state, **extra}


well_descriptors = pd.DataFrame([build_well_descriptor(well) for well in well_data])
well_descriptors = well_descriptors.merge(folds, on="wellname", how="left")
print(well_descriptors.shape)
well_descriptors.head()

C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


... (33 outputs left out)

C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


(773, 34)


,wellname,known_row_count,known_tvt_last,known_tvt_slope,known_gr_coverage,known_md_span,known_xy_span,known_z_drop,alignment_score_best,alignment_score_second,...,target_start_gap,selected_analog_distance,extrapolation_stress_score,analog_stress_score,combined_stress_score,descriptor_cluster,descriptor_cluster_fold,analog_distance_fold,extrapolation_fold,strict_hybrid_fold
0,000d7d20,1442.0,11747.37,0.354858,0.693481,1441.0,1189.689817,-476.51,0.608216,0.519653,...,-21.29,0.316157,0.268111,0.241915,0.234039,1,1,1,1,2
1,00bbac68,1545.0,12223.54,0.529087,0.928803,1544.0,1077.559777,-736.48,0.727613,0.695308,...,-35.23,0.416028,0.685640,0.561449,0.642173,3,1,2,3,1
2,00e12e8b,2083.0,11604.82,0.479654,0.920307,2082.0,1453.191016,-968.90,0.562193,0.543315,...,-17.77,0.578991,0.379204,0.865459,0.665273,2,2,3,1,1
3,015fe0d2,1654.0,12233.78,0.294325,0.917170,1653.0,1415.574133,-517.29,0.493005,0.468214,...,-8.27,0.728474,0.560964,0.957309,0.608190,4,3,3,2,3
4,01869cd4,1293.0,11745.94,0.300054,0.669760,1292.0,1103.679180,-468.21,0.654939,0.608290,...,-17.19,0.803998,0.726876,0.978008,0.674054,6,2,3,3,2


In [4]:
"Build fold-specific row feature matrices."

BASE_ROW_COLS = [
    "row_idx_original", "MD", "X", "Y", "Z", "GR_safe", "md_from_start", "md_norm",
    "distance_from_boundary", "distance_log1p", "eval_progress", "progress_squared",
    "dX", "dY", "dZ", "horizontal_step", "horizontal_dist", "z_slope_local",
    "last_known_tvt", "last_known_md", "projected_delta_10", "abs_projected_delta_10",
]

STRAT_COLS = [
    "known_row_count", "known_tvt_last", "known_tvt_slope", "known_gr_coverage", "known_md_span",
    "known_xy_span", "known_z_drop", "alignment_score_best", "alignment_score_second",
    "alignment_margin", "alignment_entropy", "typewell_best_lag", "typewell_best_corr",
    "typewell_second_corr", "typewell_best_dtw", "typewell_window_position", "target_row_count",
    "target_md_span", "endpoint_gap_to_target",
]

ANALOG_COLS = [
    "nearest_analog_distance", "nearest_analog_known_tvt_slope", "nearest_analog_alignment_margin",
    "nearest_analog_typewell_dtw", "analog_descriptor_delta", "analog_target_span_proxy",
]


def make_row_features(well, descriptor, analog_features=None):
    df = _sample_target_rows(_target_rows(well_data[well]["horizontal"]))
    if df.empty:
        return pd.DataFrame()
    horizontal = well_data[well]["horizontal"]
    known = _known_frame(horizontal)
    last_known = known.iloc[-1] if not known.empty else horizontal.iloc[0]
    out = pd.DataFrame(index=df.index)
    out["wellname"] = well
    out["row_idx_original"] = df["row_idx"].astype(float)
    out["target"] = df["TVT"].astype(float)
    out["MD"] = df["MD"].astype(float)
    out["X"] = df["X"].astype(float)
    out["Y"] = df["Y"].astype(float)
    out["Z"] = df["Z"].astype(float)
    out["GR_safe"] = _safe_gr(df["GR"])
    out["md_from_start"] = out["MD"] - float(horizontal["MD"].iloc[0])
    out["md_norm"] = out["md_from_start"] / max(float(horizontal["MD"].iloc[-1] - horizontal["MD"].iloc[0]), 1.0)
    out["distance_from_boundary"] = out["MD"] - float(last_known["MD"])
    out["distance_log1p"] = np.log1p(np.maximum(out["distance_from_boundary"], 0.0))
    denom = max(float(df["MD"].iloc[-1] - df["MD"].iloc[0]), 1.0)
    out["eval_progress"] = (out["MD"] - float(df["MD"].iloc[0])) / denom
    out["progress_squared"] = out["eval_progress"] ** 2
    out["dX"] = df["X"].diff().fillna(0.0).astype(float)
    out["dY"] = df["Y"].diff().fillna(0.0).astype(float)
    out["dZ"] = df["Z"].diff().fillna(0.0).astype(float)
    out["horizontal_step"] = np.sqrt(out["dX"] ** 2 + out["dY"] ** 2)
    out["horizontal_dist"] = out["horizontal_step"].cumsum()
    out["z_slope_local"] = out["dZ"] / np.maximum(out["horizontal_step"], 1.0)
    out["last_known_tvt"] = float(last_known.get("TVT_input", 0.0))
    out["last_known_md"] = float(last_known.get("MD", 0.0))
    if len(known) >= 11:
        recent = known.tail(11)
        md_span = max(float(recent["MD"].iloc[-1] - recent["MD"].iloc[0]), 1.0)
        projected = float(recent["TVT_input"].iloc[-1] - recent["TVT_input"].iloc[0]) / md_span
    else:
        projected = 0.0
    out["projected_delta_10"] = projected
    out["abs_projected_delta_10"] = abs(projected)
    for key in STRAT_COLS:
        out[key] = float(descriptor.get(key, 0.0))
    analog_features = analog_features or {}
    for key in ANALOG_COLS:
        out[key] = float(analog_features.get(key, 0.0))
    for col in FORMATION_COLS:
        if col in df.columns:
            out[f"dist_to_{col}"] = df[col].astype(float)
            out[f"abs_dist_to_{col}"] = np.abs(df[col].astype(float))
    return out


def build_analog_features(train_wells, valid_wells):
    descriptor_cols = [
        "known_tvt_last", "known_tvt_slope", "known_gr_coverage", "known_md_span", "known_xy_span",
        "known_z_drop", "alignment_margin", "typewell_best_corr", "typewell_best_dtw",
        "target_md_span", "endpoint_gap_to_target",
    ]
    desc = well_descriptors.set_index("wellname")
    train_matrix = desc.loc[train_wells, descriptor_cols].astype(float).replace([np.inf, -np.inf], np.nan)
    train_matrix = train_matrix.fillna(train_matrix.median()).fillna(0.0)
    scale = train_matrix.std().replace(0.0, 1.0)
    train_z = (train_matrix - train_matrix.mean()) / scale
    result = {}
    for well in valid_wells:
        row = desc.loc[[well], descriptor_cols].astype(float).replace([np.inf, -np.inf], np.nan)
        row = row.fillna(train_matrix.median()).fillna(0.0)
        row_z = (row - train_matrix.mean()) / scale
        distances = np.sqrt(((train_z - row_z.iloc[0]) ** 2).sum(axis=1))
        analog = distances.idxmin()
        analog_desc = desc.loc[analog]
        own_desc = desc.loc[well]
        result[well] = {
            "nearest_analog_distance": float(distances.loc[analog]),
            "nearest_analog_known_tvt_slope": float(analog_desc.get("known_tvt_slope", 0.0)),
            "nearest_analog_alignment_margin": float(analog_desc.get("alignment_margin", 0.0)),
            "nearest_analog_typewell_dtw": float(analog_desc.get("typewell_best_dtw", 0.0)),
            "analog_descriptor_delta": float(abs(analog_desc.get("known_tvt_last", 0.0) - own_desc.get("known_tvt_last", 0.0))),
            "analog_target_span_proxy": float(analog_desc.get("target_tvt_span", analog_desc.get("target_md_span", 0.0))),
        }
    return result


def build_fold_frame(fold_col, fold_value):
    valid_wells = well_descriptors.loc[well_descriptors[fold_col] == fold_value, "wellname"].tolist()
    train_wells = well_descriptors.loc[well_descriptors[fold_col] != fold_value, "wellname"].tolist()
    analog_valid = build_analog_features(train_wells, valid_wells)
    analog_train = build_analog_features(train_wells, train_wells)
    frames = []
    for well in train_wells:
        descriptor = well_descriptors.set_index("wellname").loc[well].to_dict()
        frame = make_row_features(well, descriptor, analog_train.get(well, {}))
        frame["split"] = "train"
        frames.append(frame)
    for well in valid_wells:
        descriptor = well_descriptors.set_index("wellname").loc[well].to_dict()
        frame = make_row_features(well, descriptor, analog_valid.get(well, {}))
        frame["split"] = "valid"
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)


## 4. Feature Family Definitions

In [5]:
"Define feature sets and model consumers."

FEATURE_SETS = {
    "safe_base": BASE_ROW_COLS,
    "strat_known_state": BASE_ROW_COLS + ["known_row_count", "known_tvt_last", "known_tvt_slope", "known_gr_coverage", "known_md_span", "known_xy_span", "known_z_drop"],
    "strat_alignment_uncertainty": BASE_ROW_COLS + STRAT_COLS,
    "strat_alignment_plus_boundary": BASE_ROW_COLS + STRAT_COLS + ["distance_from_boundary", "eval_progress", "progress_squared"],
    "strat_full": BASE_ROW_COLS + STRAT_COLS + ANALOG_COLS,
    "oracle_direct_formations": BASE_ROW_COLS + STRAT_COLS + ANALOG_COLS + [f"dist_to_{c}" for c in FORMATION_COLS] + [f"abs_dist_to_{c}" for c in FORMATION_COLS],
}

MODEL_SPECS = ["huber_delta", "hist_gradient"]
if XGB_AVAILABLE:
    MODEL_SPECS.insert(0, "xgb_squarederror")

print(FEATURE_SETS.keys())
print(MODEL_SPECS)

dict_keys(['safe_base', 'strat_known_state', 'strat_alignment_uncertainty', 'strat_alignment_plus_boundary', 'strat_full', 'oracle_direct_formations'])
['xgb_squarederror', 'huber_delta', 'hist_gradient']


## 5. Strict-CV Model Consumers

In [6]:
"Train and score candidate feature families under fold-safe validation."


def rmse(y_true, y_pred):
    return float(mean_squared_error(y_true, y_pred) ** 0.5)


def fit_predict(model_name, X_train, y_train, X_valid):
    if model_name == "xgb_squarederror" and XGB_AVAILABLE:
        model = XGBRegressor(
            objective="reg:squarederror",
            n_estimators=350,
            learning_rate=0.04,
            max_depth=4,
            min_child_weight=100,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=8.0,
            random_state=RANDOM_STATE,
            n_jobs=4,
            tree_method="hist",
        )
        model.fit(X_train, y_train, verbose=False)
        return model.predict(X_valid), model
    if model_name == "huber_delta":
        model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), HuberRegressor(alpha=0.0001, epsilon=1.35, max_iter=300))
        model.fit(X_train, y_train)
        return model.predict(X_valid), model
    if model_name == "hist_gradient":
        model = HistGradientBoostingRegressor(
            loss="squared_error",
            learning_rate=0.04,
            max_iter=350,
            max_leaf_nodes=31,
            l2_regularization=1.0,
            random_state=RANDOM_STATE,
        )
        model.fit(X_train, y_train)
        return model.predict(X_valid), model
    raise ValueError(model_name)


fold_metric_rows = []
prediction_rows = []
importance_rows = []

for fold_col in fold_cols:
    for fold_value in sorted(well_descriptors[fold_col].dropna().unique()):
        fold_frame = build_fold_frame(fold_col, fold_value)
        train_df = fold_frame[fold_frame["split"] == "train"].copy()
        valid_df = fold_frame[fold_frame["split"] == "valid"].copy()
        y_train = train_df["target"].to_numpy(dtype=float)
        y_valid = valid_df["target"].to_numpy(dtype=float)
        hold_pred = valid_df["last_known_tvt"].to_numpy(dtype=float)
        fold_metric_rows.append({
            "fold_column": fold_col,
            "fold": int(fold_value),
            "model": "last_known_hold",
            "feature_set": "baseline",
            "rmse": rmse(y_valid, hold_pred),
            "mae": float(mean_absolute_error(y_valid, hold_pred)),
            "r2": float(r2_score(y_valid, hold_pred)),
            "n_valid_rows": int(len(valid_df)),
        })
        for feature_set, columns in FEATURE_SETS.items():
            available = list(dict.fromkeys(col for col in columns if col in train_df.columns and col in valid_df.columns))
            if feature_set == "oracle_direct_formations" and not all(f"dist_to_{c}" in train_df for c in FORMATION_COLS):
                continue
            X_train = train_df[available].replace([np.inf, -np.inf], np.nan).fillna(0.0)
            X_valid = valid_df[available].replace([np.inf, -np.inf], np.nan).fillna(0.0)
            for model_name in MODEL_SPECS:
                pred, model = fit_predict(model_name, X_train, y_train, X_valid)
                score = rmse(y_valid, pred)
                fold_metric_rows.append({
                    "fold_column": fold_col,
                    "fold": int(fold_value),
                    "model": model_name,
                    "feature_set": feature_set,
                    "rmse": score,
                    "mae": float(mean_absolute_error(y_valid, pred)),
                    "r2": float(r2_score(y_valid, pred)),
                    "n_valid_rows": int(len(valid_df)),
                    "n_features": int(len(available)),
                })
                sample = valid_df[["wellname", "row_idx_original", "target", "eval_progress", "nearest_analog_distance", "alignment_margin", "typewell_best_dtw"]].copy()
                sample["fold_column"] = fold_col
                sample["fold"] = int(fold_value)
                sample["model"] = model_name
                sample["feature_set"] = feature_set
                sample["prediction"] = pred
                sample["abs_error"] = np.abs(sample["target"] - sample["prediction"])
                prediction_rows.append(sample)
                if model_name == "xgb_squarederror" and hasattr(model, "feature_importances_"):
                    for col, value in zip(available, model.feature_importances_):
                        importance_rows.append({"fold_column": fold_col, "fold": int(fold_value), "feature_set": feature_set, "model": model_name, "feature": col, "importance": float(value)})
        print(f"completed {fold_col} fold={fold_value} rows train={len(train_df)} valid={len(valid_df)}")

fold_metrics = pd.DataFrame(fold_metric_rows)
predictions = pd.concat(prediction_rows, ignore_index=True)
feature_importance = pd.DataFrame(importance_rows)
fold_metrics.head()

C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_huber.py:343: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


completed strict_hybrid_fold fold=1 rows train=46080 valid=23490


C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_huber.py:343: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_huber.py:343: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_huber.py:343: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_huber.py:343: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


... (8 outputs left out)

completed analog_distance_fold fold=2 rows train=46440 valid=23130


C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_huber.py:343: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


completed analog_distance_fold fold=3 rows train=46350 valid=23220


,fold_column,fold,model,feature_set,rmse,mae,r2,n_valid_rows,n_features
0,strict_hybrid_fold,1,last_known_hold,baseline,16.416220,11.154364,0.999225,23490,NaN
1,strict_hybrid_fold,1,xgb_squarederror,safe_base,21.543150,15.051630,0.998665,23490,22.0
2,strict_hybrid_fold,1,huber_delta,safe_base,15.789406,10.921807,0.999283,23490,22.0
3,strict_hybrid_fold,1,hist_gradient,safe_base,19.602610,13.660544,0.998895,23490,22.0
4,strict_hybrid_fold,1,xgb_squarederror,strat_known_state,20.356889,13.760594,0.998808,23490,29.0


## 6. Stress-Slice Diagnostics

In [7]:
"Summarize global, fold, and stress-slice performance."

summary = (
    fold_metrics[fold_metrics["fold_column"] == "strict_hybrid_fold"]
    .groupby(["model", "feature_set"], as_index=False)
    .agg(rmse_mean=("rmse", "mean"), rmse_std=("rmse", "std"), mae_mean=("mae", "mean"), n_valid_rows=("n_valid_rows", "sum"), n_features=("n_features", "max"))
    .sort_values("rmse_mean")
)
summary["delta_vs_current_public_best"] = summary["rmse_mean"] - CURRENT_PUBLIC_BEST
summary["delta_vs_local_safe_xgb_reference"] = summary["rmse_mean"] - LOCAL_SAFE_XGB_REFERENCE
summary["is_oracle"] = summary["feature_set"].eq("oracle_direct_formations")
safe_base = summary[summary["feature_set"].eq("safe_base")][["model", "rmse_mean"]].rename(columns={"rmse_mean": "same_model_safe_base_rmse"})
summary = summary.merge(safe_base, on="model", how="left")
summary["delta_vs_same_model_safe_base"] = summary["rmse_mean"] - summary["same_model_safe_base_rmse"]
summary = summary.sort_values(["is_oracle", "rmse_mean"]).reset_index(drop=True)

strict_predictions = predictions[predictions["fold_column"] == "strict_hybrid_fold"].copy()
base_key = (strict_predictions["model"].eq("xgb_squarederror") if XGB_AVAILABLE else strict_predictions["model"].eq("hist_gradient")) & strict_predictions["feature_set"].eq("safe_base")
base_errors = strict_predictions.loc[base_key, ["wellname", "row_idx_original", "abs_error"]].rename(columns={"abs_error": "base_abs_error"})
strict_predictions = strict_predictions.merge(base_errors, on=["wellname", "row_idx_original"], how="left")
strict_predictions["error_delta_vs_base"] = strict_predictions["abs_error"] - strict_predictions["base_abs_error"]

slice_rows = []
for (model, feature_set), group in strict_predictions.groupby(["model", "feature_set"]):
    for col, label in [("nearest_analog_distance", "high_analog_distance"), ("typewell_best_dtw", "high_typewell_dtw"), ("alignment_margin", "low_alignment_margin"), ("eval_progress", "late_progress")]:
        if col == "alignment_margin":
            threshold = group[col].quantile(0.25)
            sliced = group[group[col] <= threshold]
        else:
            threshold = group[col].quantile(0.75)
            sliced = group[group[col] >= threshold]
        if sliced.empty:
            continue
        slice_rows.append({
            "model": model,
            "feature_set": feature_set,
            "slice": label,
            "threshold": float(threshold),
            "n_rows": int(len(sliced)),
            "mae": float(sliced["abs_error"].mean()),
            "mean_delta_vs_base_abs_error": float(sliced["error_delta_vs_base"].mean()),
            "improvement_share_vs_base": float((sliced["error_delta_vs_base"] < 0).mean()),
        })

stress_slices = pd.DataFrame(slice_rows)
summary.head(12)

,model,feature_set,rmse_mean,rmse_std,mae_mean,n_valid_rows,n_features,delta_vs_current_public_best,delta_vs_local_safe_xgb_reference,is_oracle
11,huber_delta,strat_known_state,14.992214,0.654752,10.617829,69570,29.0,0.520214,1.574414,False
8,huber_delta,strat_alignment_plus_boundary,15.021553,0.638091,10.638764,69570,41.0,0.549553,1.603753,False
9,huber_delta,strat_alignment_uncertainty,15.021553,0.638091,10.638764,69570,41.0,0.549553,1.603753,False
7,huber_delta,safe_base,15.031270,0.656899,10.618797,69570,22.0,0.559270,1.613470,False
10,huber_delta,strat_full,15.108722,0.659713,10.711619,69570,47.0,0.636722,1.690922,False
12,last_known_hold,baseline,15.567125,0.762498,10.889868,69570,NaN,1.095125,2.149325,False
1,hist_gradient,safe_base,19.219185,1.509836,13.416704,69570,22.0,4.747185,5.801385,False
5,hist_gradient,strat_known_state,19.346131,1.305621,13.462301,69570,29.0,4.874131,5.928331,False
2,hist_gradient,strat_alignment_plus_boundary,19.666802,1.417919,13.477583,69570,41.0,5.194802,6.249002,False
3,hist_gradient,strat_alignment_uncertainty,19.666802,1.417919,13.477583,69570,41.0,5.194802,6.249002,False


## 7. Artifact Export

In [8]:
"Write tables, report, and run provenance."

summary_path = ARTIFACT_TABLES / f"model__stratigraphic_state_feature_screen_summary__{DATE_STAMP}.csv"
fold_path = ARTIFACT_TABLES / f"model__stratigraphic_state_feature_screen_fold_metrics__{DATE_STAMP}.csv"
stress_path = ARTIFACT_TABLES / f"model__stratigraphic_state_feature_screen_stress_slices__{DATE_STAMP}.csv"
importance_path = ARTIFACT_TABLES / f"model__stratigraphic_state_feature_screen_feature_importance__{DATE_STAMP}.csv"
pred_sample_path = ARTIFACT_TABLES / f"model__stratigraphic_state_feature_screen_prediction_sample__{DATE_STAMP}.csv"
report_path = ARTIFACT_REPORTS / f"model__stratigraphic_state_feature_screen__{DATE_STAMP}.md"

summary.to_csv(summary_path, index=False)
fold_metrics.to_csv(fold_path, index=False)
stress_slices.to_csv(stress_path, index=False)
feature_importance.to_csv(importance_path, index=False)
predictions.sample(min(len(predictions), 50000), random_state=RANDOM_STATE).to_csv(pred_sample_path, index=False)

non_oracle = summary[~summary["is_oracle"]].copy()
best = non_oracle.iloc[0].to_dict()
best_oracle = summary[summary["is_oracle"]].iloc[0].to_dict() if summary["is_oracle"].any() else None

lines = [
    "# Stratigraphic State Feature Screen",
    "",
    f"Run ID: `{RUN_ID}`",
    f"XGBoost available: `{XGB_AVAILABLE}`",
    "",
    "## Best Non-Oracle Candidate",
    "",
    f"- Model: `{best['model']}`",
    f"- Feature set: `{best['feature_set']}`",
    f"- strict_hybrid_fold RMSE: **{best['rmse_mean']:.4f}**",
    f"- Delta vs same-model safe base: **{best['delta_vs_same_model_safe_base']:.4f}**",
    f"- Delta vs local safe XGBoost reference 13.4178: **{best['delta_vs_local_safe_xgb_reference']:.4f}**",
    "",
]
if best_oracle:
    lines.extend([
        "## Oracle Diagnostic",
        "",
        f"- Best oracle model: `{best_oracle['model']}` / `{best_oracle['feature_set']}`",
        f"- Oracle strict_hybrid_fold RMSE: **{best_oracle['rmse_mean']:.4f}**",
        "",
    ])
lines.extend([
    "## Interpretation",
    "",
    "This screen tests whether known-zone stratigraphic descriptors recover formation-state signal without using hidden-test-unsafe formation columns. Promote only if non-oracle gains beat the existing safe XGBoost reference and improve stress slices. Same-model deltas are tracked separately because this notebook is a feature screen, not the final Kaggle package recipe.",
])
report_path.write_text("\n".join(lines), encoding="utf-8")

config = {
    "run_id": RUN_ID,
    "date": DATE_STAMP,
    "max_target_rows_per_well": MAX_TARGET_ROWS_PER_WELL,
    "sequence_length": SEQUENCE_LENGTH,
    "typewell_windows": TYPEWELL_WINDOWS,
    "fold_columns": fold_cols,
    "feature_sets": {key: value for key, value in FEATURE_SETS.items()},
    "model_specs": MODEL_SPECS,
    "outputs": [str(summary_path), str(fold_path), str(stress_path), str(importance_path), str(pred_sample_path), str(report_path)],
}
(ARTIFACT_LOGS / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
(ARTIFACT_LOGS / "metrics.json").write_text(json.dumps({"best_non_oracle_rmse": float(best["rmse_mean"])}, indent=2), encoding="utf-8")
(ARTIFACT_LOGS / "notes.md").write_text(report_path.read_text(encoding="utf-8"), encoding="utf-8")

print(summary_path)
print(report_path)
print(summary.head(12).to_string(index=False))

C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\tables\model__stratigraphic_state_feature_screen_summary__2026-06-30.csv
C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\reports\model__stratigraphic_state_feature_screen__2026-06-30.md
          model                   feature_set  rmse_mean  rmse_std  mae_mean  n_valid_rows  n_features  delta_vs_current_public_best  delta_vs_local_safe_xgb_reference  is_oracle
    huber_delta             strat_known_state  14.992214  0.654752 10.617829         69570        29.0                      0.520214                           1.574414      False
    huber_delta strat_alignment_plus_boundary  15.021553  0.638091 10.638764         69570        41.0                      0.549553                           1.603753      False
    huber_delta   strat_alignment_uncertainty  15.021553  0.638091 10.638764         69570        41.0                      0.549553                           1.60

## 8. Findings

In [9]:
"Display the headline tables for notebook readers."

print("Best strict-hybrid candidates:")
display(summary.head(15))
print("Stress slices for the best feature set/model:")
display(stress_slices[(stress_slices["model"] == best["model"]) & (stress_slices["feature_set"] == best["feature_set"])] )
if not feature_importance.empty:
    display(feature_importance.groupby("feature", as_index=False)["importance"].mean().sort_values("importance", ascending=False).head(20))

Best strict-hybrid candidates:


,model,feature_set,rmse_mean,rmse_std,mae_mean,n_valid_rows,n_features,delta_vs_current_public_best,delta_vs_local_safe_xgb_reference,is_oracle
11,huber_delta,strat_known_state,14.992214,0.654752,10.617829,69570,29.0,0.520214,1.574414,False
8,huber_delta,strat_alignment_plus_boundary,15.021553,0.638091,10.638764,69570,41.0,0.549553,1.603753,False
9,huber_delta,strat_alignment_uncertainty,15.021553,0.638091,10.638764,69570,41.0,0.549553,1.603753,False
7,huber_delta,safe_base,15.031270,0.656899,10.618797,69570,22.0,0.559270,1.613470,False
10,huber_delta,strat_full,15.108722,0.659713,10.711619,69570,47.0,0.636722,1.690922,False
12,last_known_hold,baseline,15.567125,0.762498,10.889868,69570,NaN,1.095125,2.149325,False
1,hist_gradient,safe_base,19.219185,1.509836,13.416704,69570,22.0,4.747185,5.801385,False
5,hist_gradient,strat_known_state,19.346131,1.305621,13.462301,69570,29.0,4.874131,5.928331,False
2,hist_gradient,strat_alignment_plus_boundary,19.666802,1.417919,13.477583,69570,41.0,5.194802,6.249002,False
3,hist_gradient,strat_alignment_uncertainty,19.666802,1.417919,13.477583,69570,41.0,5.194802,6.249002,False


Stress slices for the best feature set/model:


,model,feature_set,slice,threshold,n_rows,mae,mean_delta_vs_base_abs_error,improvement_share_vs_base
44,huber_delta,strat_known_state,high_analog_distance,1.751811,17460,10.823129,-6.242653,0.643528
45,huber_delta,strat_known_state,high_typewell_dtw,0.901067,17460,9.833367,-4.822473,0.643700
46,huber_delta,strat_known_state,low_alignment_margin,0.020603,17460,10.750230,-5.018878,0.656586
47,huber_delta,strat_known_state,late_progress,0.752708,17393,13.590315,-4.026804,0.565285


,feature,importance
41,last_known_tvt,0.571382
36,known_tvt_last,0.448216
4,Z,0.043159
40,last_known_md,0.005882
5,abs_dist_to_ANCC,0.003658
2,X,0.001552
7,abs_dist_to_ASTNU,0.001428
23,dist_to_ASTNU,0.000788
3,Y,0.000705
26,dist_to_EGFDU,0.000658
